In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [18]:
model_gpt = ChatOpenAI(
    model = 'gpt-3.5-turbo'
)

# Simple chat model

In [19]:
temp = PromptTemplate(
    template = 'Give me 2 line on {topic}',
    input_variables = ['topic']
)

prompt = temp.invoke({'topic' : 'India'})

In [21]:
model_gpt.invoke(prompt).content

'India is a land of diversity, where languages, cultures, and traditions blend in harmony. From the majestic Himalayas to the serene backwaters of Kerala, its beauty captures the heart of all who visit.'

In [27]:
# Using Chain
from langchain.chains import LLMChain

In [26]:
chain = temp | model_gpt

print(chain.invoke('India').content)

India, land of diversity and culture,
Where tradition and modernity coexist in harmony.


# PDF Reader

In [82]:

from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings   # We will use this to generate embeddings
from langchain.vectorstores import FAISS

In [40]:
loader = TextLoader(r"00 Gaargi and Yaagyavalakya.txt", encoding = 'utf-8')
loader

In [67]:
documents = loader.load()
print(documents[0].page_content)    # .page_content shows only content and excludes the meta data

# TextLoader().load() reads the whole txt file into one document. 
# It doesn't split the documents into pages, it just stores into one single doc in the form of string

The most fascinating philosophical dialogues from the Brihadaranyaka Upanishad (Yajnavalkya–Gargi Samvada).
This debate is not just history — it’s part of India’s ancient intellectual tradition, where women like Gargi Vachaknavi were revered as profound scholars of Vedic philosophy.

Let me narrate the full context, sequence, and philosophical essence for you.

1. Context of the Debate
The event takes place in the court of King Janaka of Videha, who was a patron of knowledge and philosophy.

King Janaka organized a brahmayajña (philosophical assembly) to discuss the nature of Brahman — the ultimate reality.

The hall was filled with the era’s greatest scholars, sages, and philosophers, including Yajnavalkya, a celebrated Vedic rishi, and Gargi Vachaknavi, one of the most learned women of her time.

The winner of the debate would receive a thousand cows, each with gold ornaments on their horns.

2. Yajnavalkya’s Early Exchanges
Before Gargi speaks, Yajnavalkya is challenged by several s

In [61]:
#Splitting the whole doc into different small docs

text_splitter  = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 50)
pages = text_splitter.split_documents(documents) 
print(len(pages))

10


In [60]:
print(pages[0])    # pages[0] prints everything, content + metadata
print()
print()

print(pages[5].page_content)    # pages[5].page_cpntent prints only content of the page

page_content='The most fascinating philosophical dialogues from the Brihadaranyaka Upanishad (Yajnavalkya–Gargi Samvada).
This debate is not just history — it’s part of India’s ancient intellectual tradition, where women like Gargi Vachaknavi were revered as profound scholars of Vedic philosophy.

Let me narrate the full context, sequence, and philosophical essence for you.

1. Context of the Debate
The event takes place in the court of King Janaka of Videha, who was a patron of knowledge and philosophy.' metadata={'source': '00 Gaargi and Yaagyavalakya.txt'}


Those in the worlds of the Gods.

Those in the worlds of Indra.

Those in the worlds of Prajapati.

Finally, those are woven in Brahman — the immutable, imperishable.

6. The Deeper Second Question
Now Gargi makes her famous second challenge.

Gargi:

“O Yajnavalkya, that immutable, in which are woven the sky and the earth, and in which the world-order (dharma) and the past, present, and future are woven — tell me of it.”

Yajna

In [86]:
# Convert the pages into vector and storing it

vectorstore = FAISS.from_documents(pages, OpenAIEmbeddings())

In [89]:
# Creating a retriever
retriever = vector.as_retriever()

In [100]:
# Retrieve related documents.
# This will give us all the related documents from the total documents

query = 'Who was Gaargi and what was the objective of this debate'

retrieved_doc = retriever.invoke(query)

print(retrieved_doc)   # This prints all the retrieved documents from the total documents

[Document(id='11bcb445-4f16-45d6-aa38-4a8183e38c1a', metadata={'source': '00 Gaargi and Yaagyavalakya.txt'}, page_content='The most fascinating philosophical dialogues from the Brihadaranyaka Upanishad (Yajnavalkya–Gargi Samvada).\nThis debate is not just history — it’s part of India’s ancient intellectual tradition, where women like Gargi Vachaknavi were revered as profound scholars of Vedic philosophy.\n\nLet me narrate the full context, sequence, and philosophical essence for you.\n\n1. Context of the Debate\nThe event takes place in the court of King Janaka of Videha, who was a patron of knowledge and philosophy.'), Document(id='4f2a9458-b681-4f52-8257-82d2ca12d441', metadata={'source': '00 Gaargi and Yaagyavalakya.txt'}, page_content='2. Yajnavalkya’s Early Exchanges\nBefore Gargi speaks, Yajnavalkya is challenged by several scholars, such as Ashwala, Uddalaka, and Buddha (not the Buddha of Buddhism, but another sage).\n\nYajnavalkya defeats them with his profound knowledge.\n\nGa

In [102]:
# Combine all the retrieved documents into a single document

retrieved_text = '\n'.join([doc.page_content for doc in retrieved_doc])

retrieved_text

'The most fascinating philosophical dialogues from the Brihadaranyaka Upanishad (Yajnavalkya–Gargi Samvada).\nThis debate is not just history — it’s part of India’s ancient intellectual tradition, where women like Gargi Vachaknavi were revered as profound scholars of Vedic philosophy.\n\nLet me narrate the full context, sequence, and philosophical essence for you.\n\n1. Context of the Debate\nThe event takes place in the court of King Janaka of Videha, who was a patron of knowledge and philosophy.\n2. Yajnavalkya’s Early Exchanges\nBefore Gargi speaks, Yajnavalkya is challenged by several scholars, such as Ashwala, Uddalaka, and Buddha (not the Buddha of Buddhism, but another sage).\n\nYajnavalkya defeats them with his profound knowledge.\n\nGargi, witnessing this, decides to question him — but not with ordinary queries.\n\n3. Gargi Steps Forward\nGargi Vachaknavi was not only a Brahmavadini (woman knower of Brahman) but also fearless.\n\nShe first warns the assembly:\nShe first warns 

In [108]:
retrieved_doc[3].page_content

'Gender equality in learning: This debate shows that in Vedic times, women could be esteemed philosophers.\n\nMetaphysical ladder: Gargi’s “woven in” questioning is like peeling layers of reality to reach the ultimate substratum.'

In [111]:
prompt = f'''Based on the below given text, answer this question- {query}
{retrieved_text}'''

print(prompt)

Based on the below given text, answer this question- Who was Gaargi and what was the objective of this debate
The most fascinating philosophical dialogues from the Brihadaranyaka Upanishad (Yajnavalkya–Gargi Samvada).
This debate is not just history — it’s part of India’s ancient intellectual tradition, where women like Gargi Vachaknavi were revered as profound scholars of Vedic philosophy.

Let me narrate the full context, sequence, and philosophical essence for you.

1. Context of the Debate
The event takes place in the court of King Janaka of Videha, who was a patron of knowledge and philosophy.
2. Yajnavalkya’s Early Exchanges
Before Gargi speaks, Yajnavalkya is challenged by several scholars, such as Ashwala, Uddalaka, and Buddha (not the Buddha of Buddhism, but another sage).

Yajnavalkya defeats them with his profound knowledge.

Gargi, witnessing this, decides to question him — but not with ordinary queries.

3. Gargi Steps Forward
Gargi Vachaknavi was not only a Brahmavadini (

In [112]:
response = model_gpt.invoke(prompt)

print(response.content)

Gaargi was a revered scholar of Vedic philosophy and a Brahmavadini (woman knower of Brahman). The objective of this debate was to challenge Yajnavalkya, a reputed sage, with profound questions about cosmology and metaphysics, showcasing the intellectual prowess of women in the ancient Indian tradition.


# Chained process for both Simple model and Retrieval Model

# `1) Simple LLM`

In [115]:
temp = PromptTemplate(
    template = 'Write 2 lines on {topic}',
    imput_variables = ['topic']
)

chain = temp | model_gpt
response = chain.invoke({'topic': 'India'})
print(response.content)

India is a land of diverse cultures, languages, and traditions.
From bustling cities to serene countryside, India enchants with its beauty and vibrancy.


# 2) Retrieval model

In [118]:
from langchain.chains import RetrievalQA

In [122]:
chain = RetrievalQA.from_chain_type(llm = model_gpt, retriever = retriever)
chain

RetrievalQA(verbose=False, combine_documents_chain=StuffDocumentsChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="Use the following pieces of context to answer the user's question.\nIf you don't know the answer, just say that you don't know, don't try to make up an answer.\n----------------\n{context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})]), llm=ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000016935F83B10>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000016935F81A10>, root_client=<openai.OpenAI object at 

In [123]:
print(query)

Who was Gaargi and what was the objective of this debate


In [126]:
response = chain.invoke(query)

print(response)

{'query': 'Who was Gaargi and what was the objective of this debate', 'result': 'Gargi Vachaknavi was a revered female scholar of Vedic philosophy in ancient India. The objective of the debate between Gargi and Yajnavalkya was to engage in an intellectual discourse about profound philosophical matters, such as the nature of the cosmos and ultimate reality. It showcased gender equality in learning and allowed Gargi to challenge Yajnavalkya, a renowned sage, demonstrating her expertise and fearlessness in the field of philosophy.'}


# In this session, we saw how Runnables reduces the lines of codes.

# Next Session: How Runnables work